<a href="https://colab.research.google.com/github/Farrukh776/flyrank-ai/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Farrukh776/flyrank-ai/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Row = one content item, for one client, on one calendar day (report_date × client_hash_id × content_hash_id) — the grain of fact_content_daily_performance.
Table(s): fact_content_daily_performance (month=2026-03 partition) joined to dim_content for static metadata (age, word count) and dim_clients for per-client history flags.
Time window: March 2026 only — a mid-panel month, not the final sealed month.
Target/proxy: a within-month decline flag — whether a page's impressions fell from the first half of March to the second half. (A proxy, not a true future-window label — that's Week 5+ work.)
Deliberately excluded: any product-computed flag (health_score, priority_score) — not shipped in this data on purpose, and any GA4 field for rows where ga4_data_available = FALSE.

In [ ]:
#setup

import os, subprocess, sys
if "google.colab" in sys.modules and not os.path.exists("flyrank-ai"):
    subprocess.run(["git", "clone", "https://github.com/Farrukh776/flyrank-ai.git"], check=True)
if os.path.basename(os.getcwd()) != "flyrank-ai":
    os.chdir("flyrank-ai")

%pip -q install duckdb

import duckdb
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"  # mid-panel month — never the _sample (that's the sealed final month)
print("Connected. Working month:", MONTH)

Connected. Working month: 2026-03


In [ ]:
q = f"""
SELECT COUNT(*) AS n_rows, MIN(report_date) AS min_date, MAX(report_date) AS max_date
FROM read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')
"""
con.sql(q)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬────────────┬────────────┐
│ n_rows  │  min_date  │  max_date  │
│  int64  │    date    │    date    │
├─────────┼────────────┼────────────┤
│ 9841378 │ 2026-03-01 │ 2026-03-31 │
└─────────┴────────────┴────────────┘

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Field | Bucket | Why |
|---|---|---|
| `gsc_impressions`, `gsc_clicks`, `gsc_sum_position` (→ derived `avg_position`) | Feature | observed search signals, known at prediction time |
| `ga4_engaged_sessions`, `ga4_sessions` (→ derived `engagement_rate`) | Feature | observed engagement, only valid where `ga4_data_available` is true |
| `content_created_date` (→ derived `content_age_days`), `word_count` | Feature | static content metadata from `dim_content` |
| `client_hash_id`, `content_hash_id` | Context | joining/grouping only, never a model input |
| within-month impression decline flag (`declined`) | Label/proxy | what we'd predict — a proxy, since it's within-month, not a true future outcome |
| `pct_change` | Excluded | the column the label was literally derived from — proven to leak (AUC 0.557 → 0.9997) |
| `ga4_data_available`, `gsc_data_available` | Context (filter) | tells us whether zeros are real "no engagement" or "not tracked yet" |
| any `health_score`/`priority_score`/`action_type` | Excluded | product-decision output, not shipped in this data |

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
con.sql(f"""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) c
FROM read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')
GROUP BY 1,2,3 HAVING c > 1 LIMIT 5
""")  # empty result = grain holds

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────────┬─────────────────┬───────┐
│ report_date │ client_hash_id │ content_hash_id │   c   │
│    date     │    varchar     │     varchar     │ int64 │
├─────────────┴────────────────┴─────────────────┴───────┤
│                         0 rows                         │
└────────────────────────────────────────────────────────┘

In [ ]:
q = f"""
SELECT COUNT(*) AS n_rows, MIN(report_date) AS min_date, MAX(report_date) AS max_date
FROM read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')
"""
con.sql(q)

┌─────────┬────────────┬────────────┐
│ n_rows  │  min_date  │  max_date  │
│  int64  │    date    │    date    │
├─────────┼────────────┼────────────┤
│ 9841378 │ 2026-03-01 │ 2026-03-31 │
└─────────┴────────────┴────────────┘

In [ ]:
con.sql(f"""
SELECT COUNT(*) AS total_rows,
       SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows
FROM read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')
""")

┌────────────┬────────────────────┐
│ total_rows │ ga4_available_rows │
│   int64    │       int128       │
├────────────┼────────────────────┤
│    9841378 │             413966 │
└────────────┴────────────────────┘

In [ ]:
#5 feature frame

features = con.sql(f"""
SELECT
  f.content_hash_id,
  SUM(f.gsc_impressions) AS impressions_month,
  SUM(f.gsc_clicks) AS clicks_month,
  SUM(f.gsc_sum_position) / NULLIF(SUM(f.gsc_impressions), 0) AS avg_position,
  SUM(CASE WHEN f.ga4_data_available THEN f.ga4_engaged_sessions END)
    / NULLIF(SUM(CASE WHEN f.ga4_data_available THEN f.ga4_sessions END), 0) AS engagement_rate,
  DATE '2026-03-31' - ANY_VALUE(d.content_created_date) AS content_age_days,
  ANY_VALUE(d.word_count) AS word_count
FROM read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet') f
LEFT JOIN read_parquet('{REL}/dim_content.parquet') d
  ON f.content_hash_id = d.content_hash_id
GROUP BY f.content_hash_id
""").df()
features.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,impressions_month,clicks_month,avg_position,engagement_rate,content_age_days,word_count
0,content_39d7361b4945d504,77.0,0.0,4.311688,NaN,47,3579
1,content_cec711b02f3bbde6,602.0,4.0,4.365449,NaN,47,2455
2,content_275b6f7f733016d4,810.0,1.0,4.624691,NaN,47,3653
3,content_ceaec531566ffcfc,82.0,0.0,8.097561,NaN,47,3096
4,content_755d951187fcd70a,1858.0,6.0,1.920344,NaN,47,3305


In [ ]:
import numpy as np

X_leaky = X_honest.copy()
X_leaky["pct_change"] = pd.to_numeric(merged["pct_change"], errors="coerce")
X_leaky["pct_change"] = X_leaky["pct_change"].replace([np.inf, -np.inf], np.nan).fillna(0)

leaky_score = LogisticRegression(max_iter=1000).fit(X_leaky, y).predict_proba(X_leaky)[:,1]
print("Leaky AUC (with pct_change):", roc_auc_score(y, leaky_score))

print("\nKept: honest features only (impressions_month, avg_position, content_age_days, word_count). pct_change removed.")

Leaky AUC (with pct_change): 0.9999652604477373

Kept: honest features only (impressions_month, avg_position, content_age_days, word_count). pct_change removed.


The leaky feature (pct_change) is literally what the label (declined) was computed from — AUC jumping from 0.557 to 0.9997 confirms the model isn't learning a pattern, it's just re-deriving the label. Removed; only the honest features are kept going forward.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This 30-day window can't distinguish real decline from seasonality or noise — a proper decline
definition needs a longer baseline and a persistence check (per the lane guide's Section 7).

GA4 engagement data is only available for 4.2% of March rows (9,841,378 total, 413,966 with GA4
tracking active). For the other ~96%, engagement_rate is NaN because that client's GA4 tracking
hadn't started yet — not because engagement was zero. Any engagement-based feature silently drops
almost the entire dataset unless ga4_data_available is checked explicitly first.

This is also only one month of an unbalanced panel — client history depth varies wildly across
the full warehouse, so this slice isn't representative of clients with shorter or missing history.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.